In [ ]:
# =============================
# STEP 1: Install
# =============================
!pip install transformers scikit-learn torch

# =============================
# STEP 2: Imports
# =============================
import pandas as pd
import torch
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

# =============================
# STEP 3: Upload Dataset
# =============================
from google.colab import files
files.upload()

df = pd.read_csv('IMDB Dataset.csv')


df = df.sample(800, random_state=42)

# =============================
# STEP 4: Preprocessing
# =============================
df = df.dropna()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['review'] = df['review'].apply(clean_text)

df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# =============================
# STEP 5: Split
# =============================
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42
)

val_texts = test_texts[:50]
val_labels = test_labels[:50]

test_texts = test_texts[50:]
test_labels = test_labels[50:]

# =============================
# STEP 6: Tokenization
# =============================
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN = 32

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx])
        }

train_loader = DataLoader(TextDataset(train_texts, train_labels), batch_size=8, shuffle=True)
val_loader = DataLoader(TextDataset(val_texts, val_labels), batch_size=8)
test_loader = DataLoader(TextDataset(test_texts, test_labels), batch_size=8)

# =============================
# STEP 7: Model
# =============================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

# =============================
# STEP 8: Train
# =============================
def train(model, loader):
    model.train()
    for batch in loader:
        optimizer.zero_grad()
        inputs = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        loss = model(inputs, attention_mask=mask, labels=labels).loss
        loss.backward()
        optimizer.step()

# =============================
# STEP 9: Evaluate
# =============================
def evaluate(model, loader):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in loader:
            inputs = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(inputs, attention_mask=mask)
            logits = outputs.logits

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true.extend(labels.cpu().numpy())

    acc = accuracy_score(true, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true, preds, average='binary')
    cm = confusion_matrix(true, preds)

    return acc, prec, rec, f1, cm

# =============================
# STEP 10: Run Training
# =============================
train(model, train_loader)

# =============================
# STEP 11: Final Evaluation
# =============================
acc, prec, rec, f1, cm = evaluate(model, test_loader)

print("\nResults:")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)
print("Confusion Matrix:\n", cm)